# Discrete-event systems with StormPy

In [2]:
from pathlib import Path
import json
import random

import pandas as pd
import stormpy
import stormpy.simulator
from IPython.display import Markdown, display

In [ ]:
def load_model(filename, verbose=True, prism_compat=False):
    program = stormpy.parse_prism_program(
        str(filename), prism_compat=prism_compat
    )
    options = stormpy.BuilderOptions()
    options.set_build_state_valuations(True)
    options.set_build_choice_labels(True)
    options.set_build_all_labels()
    options.set_build_all_reward_models()
    model = stormpy.build_sparse_model_with_options(program, options)
    if verbose:
        print(model)
    return model


def state_values(model, state_id):
    return json.loads(str(model.state_valuations.get_json(state_id)))


def choice_name(model, state_id, local_action):
    if not model.has_choice_labeling():
        return "—"
    first_choice = model.transition_matrix.get_row_group_start(state_id)
    labels = sorted(
        model.choice_labeling.get_labels_of_choice(first_choice + local_action)
    )
    return ", ".join(labels) if labels else "—"

In [8]:
def simulate_path(model, *, max_steps=10, seed=42, stop_labels=()):
    rng = random.Random(seed)
    simulator = stormpy.simulator.create_simulator(model, seed=seed)
    state_id, _observation, labels = simulator.restart()
    elapsed_time = 0.0
    rows = [
        {
            "step": 0,
            "time": elapsed_time
            if model.model_type == stormpy.ModelType.CTMC
            else None,
            "action": "—",
            **state_values(model, state_id),
            "labels": ", ".join(sorted(label for label in labels if label != "init")),
        }
    ]

    for step in range(1, max_steps + 1):
        if set(labels) & set(stop_labels) or simulator.is_done():
            break
        actions = list(simulator.available_actions())
        if not actions:
            break
        action = rng.choice(actions)
        action_label = choice_name(model, state_id, action)

        if model.model_type == stormpy.ModelType.CTMC:
            exit_rate = float(model.exit_rates[state_id])
            elapsed_time += rng.expovariate(exit_rate)

        state_id, _observation, labels = simulator.step(action)
        rows.append(
            {
                "step": step,
                "time": elapsed_time
                if model.model_type == stormpy.ModelType.CTMC
                else None,
                "action": action_label,
                **state_values(model, state_id),
                "labels": ", ".join(
                    sorted(label for label in labels if label != "init")
                ),
            }
        )

    trace = pd.DataFrame(rows)
    if trace["time"].isna().all():
        trace = trace.drop(columns="time")
    return trace

## DTMC: User Engagement

In [15]:
engagement_dtmc = load_model("./engagement.pm")

-------------------------------------------------------------- 
Model type: 	DTMC (sparse)
States: 	5
Transitions: 	12
Reward Models:  none
State Labels: 	9 labels
   * deadlock -> 0 item(s)
   * disengaged -> 1 item(s)
   * success -> 1 item(s)
   * converted -> 1 item(s)
   * init -> 1 item(s)
   * failure -> 1 item(s)
   * engaged -> 1 item(s)
   * browsing -> 1 item(s)
   * abandoned -> 1 item(s)
Choice Labels: 	0 labels
-------------------------------------------------------------- 



In [ ]:
simulate_path(
    engagement_dtmc, max_steps=10, seed=7, stop_labels={"converted", "abandoned"}
)

,step,action,state,labels
0,0,—,0,browsing
1,1,—,1,engaged
2,2,—,1,engaged
3,3,—,3,"converted, success"


The `state` encoding is 0 = browsing, 1 = engaged, 2 = disengaged, 3 = converted, and 4 = abandoned. Change the seed or rerun with several seeds to explore different paths. We stop when a terminal label is reached instead of displaying repeated absorbing self-loops.

## MDP: Activity Recommendation Agent

In [11]:
activity_mdp = load_model("activity_agent.pm")

-------------------------------------------------------------- 
Model type: 	MDP (sparse)
States: 	13
Transitions: 	27
Choices: 	15
Reward Models:  task_completion
State Labels: 	9 labels
   * deadlock -> 0 item(s)
   * s_success -> 1 item(s)
   * s_abandon -> 4 item(s)
   * s_W -> 1 item(s)
   * s_WM -> 1 item(s)
   * s_M -> 1 item(s)
   * init -> 1 item(s)
   * s_0 -> 1 item(s)
   * done -> 4 item(s)
Choice Labels: 	7 labels
   * ask_weather -> 2 item(s)
   * ask_both -> 1 item(s)
   * complete -> 1 item(s)
   * ask_mood -> 2 item(s)
   * terminate -> 4 item(s)
   * recommend -> 1 item(s)
   * done -> 4 item(s)
-------------------------------------------------------------- 



In [ ]:
simulate_path(activity_mdp, max_steps=10, seed=42, stop_labels={"done"})

,step,action,mood,status,weather,labels
0,0,—,0,0,0,s_0
1,1,ask_both,1,0,1,s_WM
2,2,recommend,1,1,1,s_success
3,3,complete,1,3,1,done


`weather` and `mood` use 0 = unknown and 1 = known. For `status`, 0 = active, 1 = success, 2 = abandoned, and 3 = done. The action column exposes the chosen command. Fixing an MDP policy induces a DTMC; finding and checking policies is covered in the next lecture.

## 3. CTMC: timed user engagement

`engagement_timed.pm` uses transition rates rather than one-step probabilities. We keep the standard PRISM CTMC syntax so the same file can be run by the PRISM model checker. StormPy therefore parses it using `prism_compat=True`.

> **Expected Storm warning:** Storm may say that the CTMC uses ‘probabilistic commands’ and suggest Markovian commands. In compatibility mode, Storm correctly interprets the values as PRISM CTMC rates; this warning does not indicate an invalid or incorrectly constructed model.

The simulator chooses transitions from the embedded probability distribution, while `simulate_path` samples exponentially distributed holding times using Storm's constructed exit rates.

In [13]:
engagement_ctmc = load_model("engagement_timed.pm", prism_compat=True)

WARN  (Program.cpp:240): The input model is a CTMC, but uses probabilistic commands like they are used in PRISM. Consider rewriting the commands to use Markovian commands instead.
-------------------------------------------------------------- 
Model type: 	CTMC (sparse)
States: 	5
Transitions: 	10
Reward Models:  none
State Labels: 	8 labels
   * init -> 1 item(s)
   * engaged -> 1 item(s)
   * deadlock -> 2 item(s)
   * disengaged -> 1 item(s)
   * converted -> 1 item(s)
   * done -> 2 item(s)
   * browsing -> 1 item(s)
   * abandoned -> 1 item(s)
Choice Labels: 	0 labels
-------------------------------------------------------------- 



In [14]:
simulate_path(
    engagement_ctmc, max_steps=20, seed=19, stop_labels={"converted", "abandoned"}
)

,step,time,action,state,labels
0,0,0.000000,—,0,browsing
1,1,2.364162,—,1,engaged
2,2,3.956159,—,3,"converted, deadlock, done"
